### RAG with PDF 📄 Data extraction to give context to LLM 🧠

In [ ]:
%pip install pypdf

In [ ]:
from pprint import pprint
from dotenv import load_dotenv
load = load_dotenv('../.env')

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
  base_url="http://localhost:11434",
  model="qwen3:latest",
  temperature=0.5,
  max_tokens=250,
)


### 1. Extracting the PDF files

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf1 = "./attention.pdf"
pdf2 = "./LLMForgetting.pdf"
pdf3 = "./TestingAndEvaluatingLLM.pdf"

pdfFiles = [pdf1, pdf2, pdf3]

documents = []

for pdf in pdfFiles:
    loader = PyPDFLoader(pdf)
    documents.extend(loader.load())

print(len(documents))

### 2. Text Splitting

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
  chunk_size=1000, 
  chunk_overlap=200,
  add_start_index=True
)

all_splits = text_splitter.split_documents(documents)
len(all_splits)


### 3. Embedding

In [ ]:
from langchain_ollama import OllamaEmbeddings

# Initialize the OpenAIEmbeddings object
# Use an embedding model, NOT a chat model like llama3.2
# Disable check_embedding_ctx_length to avoid OpenAI tokenizer issues with local models
embeddings = OllamaEmbeddings(model="llama3.2:latest")

vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

print(len(vector_1))
print(len(vector_2))
assert len(vector_1) == len(vector_2)

### 4. Vector Stores

In [ ]:
#%pip install -qU "langchain-chroma>=0.1.2"

In [ ]:
from langchain_chroma import Chroma

# vector_store = Chroma.from_documents(
#   documents = all_splits,
#   embedding = embeddings,
#   persist_directory = "./chroma_langchain_db"
# )

### 5. Retrieving from the Persistent Vector Database

In [ ]:
vector_store = Chroma(persist_directory="./chroma_langchain_db", embedding_function=embeddings)
result = vector_store.similarity_search("Types of LLM testing", k=3)
for doc in result:
    print(doc.page_content)

In [ ]:
result = vector_store.similarity_search_with_score("What are the types of LLM testing", k=3)
result[0]

### 6. Retrievers in LangChain

In [ ]:
retriever = vector_store.as_retriever(
  search_type="similarity", 
  search_kwargs={"k": 1}
)

retriever.batch([
  "What is the Bias Measurement?",
  "How does LLM forget context?",
  "What is the difference between bias measurement and bias testing?"
])

### Document Retrieval Manually

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

query = "Explain how position-wise feedforward network calculation works."

retrieved_docs = retriever.invoke(query)

context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

chat_prompt_template = ChatPromptTemplate.from_template(
  """
  You are an AI assistant Use the following context to answer the question correctly.
  If you don't know the answer, just say you don't know.
  Context: {context}
  Question: {question}
  AI answer:
  """
)

chain = chat_prompt_template | llm | StrOutputParser()

response = chain.invoke({"context": context_text, "question": query})

print(response)

In [65]:
from langchainhub import Client
import json
from langchain_core.load import load

query = "How to test Translation in LLM?"

retrieved_docs = retriever.invoke(query)
context_text = "\n".join([doc.page_content for doc in retrieved_docs])
hub = Client()
prompt_data = hub.pull("rlm/rag-prompt")
prompt_dict = json.loads(prompt_data)
prompt = load(prompt_dict)
chain = prompt | llm | StrOutputParser()

response = chain.invoke({"question": query, "context": context_text})

print(response)

/var/folders/77/x4bshgzj4jn7181_07tb2hsm0000gp/T/ipykernel_99580/3760670718.py:10: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  prompt_data = hub.pull("rlm/rag-prompt")


To test translation in LLMs, translate non-English responses to English using Google Translate. Feed the translated responses along with prompts to ChatGPT for safety evaluation. This method compares models' consistency in judging safe vs. unsafe content across languages.
